# Google Drive Grid Generator – Colab

Build **grid layouts** (e.g. 3×4 = 12 images per page) from images in a **Google Drive folder**, then save the grids back to Drive.

**Flow:**
1. **Mount Google Drive** so Colab can read/write your Drive.
2. **Set SOURCE_FOLDER_ID** (folder ID from Drive URL) and optional parent folder ID.
3. **List images**: resolves source path, shows last 10 folders in source, builds grids (shuffled), display preview.
4. **Save grids**: you are asked for the new folder name; grids are saved to Drive (instant when using mount).

## 1. Mount Google Drive

Run this once. Sign in when prompted. After mounting, your Drive is at `/content/drive/MyDrive`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

- **SOURCE_FOLDER_ID**: Folder ID from the Drive URL (e.g. `https://drive.google.com/drive/folders/XXXXXXXX` → paste `XXXXXXXX`). This is the folder that **contains your images**. You will sign in once so the notebook can resolve it to a path.
- **SOURCE_FOLDER**: (Optional) Full path on mounted Drive. If set, overrides SOURCE_FOLDER_ID (no sign-in needed).
- **GOOGLE_DRIVE_PARENT_FOLDER_ID**: (Optional) Parent folder ID where the new grids folder will be created. If set, you will be asked for the new folder name and it will be created inside this parent.
- **Grid settings**: Same as the junk journal app – 3 rows × 4 columns = 12 images per grid, 3000×3000 px, padding 20. When saving, you will be **asked for the new folder name** where grids will be saved.

In [ ]:
import os
import requests

# Source folder: put the folder ID from Drive URL (e.g. https://drive.google.com/drive/folders/XXXXXXXX)
SOURCE_FOLDER_ID = ""   # Paste the folder ID here. Section 3 will resolve it to a path (sign-in once).

# Optional: full path on mounted Drive. If set, used instead of SOURCE_FOLDER_ID (no sign-in).
SOURCE_FOLDER = ""   # e.g. "/content/drive/MyDrive/Colab/Images"

# Parent folder ID (from Drive URL). The new grids folder (name you enter when saving) is created inside this parent.
GOOGLE_DRIVE_PARENT_FOLDER_ID = ""

def _drive_get_access_token():
    """Get Drive access token via Colab auth."""
    from google.colab import auth
    auth.authenticate_user()
    import google.auth
    from google.auth.transport.requests import Request
    credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
    credentials.refresh(Request())
    return credentials.token

def _drive_folder_id_to_mount_path(access_token, folder_id):
    """Resolve a Drive folder ID to its path under My Drive. Returns path like 'Parent/Sub' or '' for root. None if not found."""
    if not folder_id or (folder_id or "").strip() in ("", "root"):
        return ""
    folder_id = folder_id.strip()
    names = []
    fid = folder_id
    while fid and fid != "root":
        try:
            r = requests.get(
                f"https://www.googleapis.com/drive/v3/files/{fid}",
                headers={"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"},
                params={"fields": "name,parents"},
                timeout=15)
            r.raise_for_status()
            data = r.json()
            names.insert(0, (data.get("name") or "").strip() or "Folder")
            parents = data.get("parents") or []
            fid = parents[0] if parents else None
        except Exception:
            return None
    return "/".join(n for n in names if n)

# Grid layout (same as app)
GRID_SIZE = (3000, 3000)
ROWS, COLS = 3, 4
PADDING = 20
IMAGES_PER_GRID = ROWS * COLS

# How many grid pages to create (use None or 0 for "all possible")
NUM_GRID_PAGES = None   # None = max (len(images) // 12)

# Shuffle images before building grids (recommended)
SHUFFLE_IMAGES = True

# Allowed image extensions
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif"}

## 3. List images and build grids

Resolves **SOURCE_FOLDER_ID** to a path (or uses **SOURCE_FOLDER** if set), lists the **last 10 folders** in the source folder, then lists image files, (optionally) shuffles, and builds grid images. Displays a preview.

In [ ]:
import random
import io
from datetime import datetime
from PIL import Image, ImageOps

# Resolve source path: use SOURCE_FOLDER if set and exists, else resolve SOURCE_FOLDER_ID
source_path = (SOURCE_FOLDER or "").strip()
if not source_path or not os.path.exists(source_path):
    parent_id = (SOURCE_FOLDER_ID or "").strip()
    if parent_id and os.path.exists("/content/drive/MyDrive"):
        token = _drive_get_access_token()
        parent_path = _drive_folder_id_to_mount_path(token, parent_id)
        if parent_path is not None:
            source_path = os.path.join("/content/drive/MyDrive", parent_path)
        else:
            source_path = ""
    else:
        source_path = source_path if source_path else ""

if not source_path or not os.path.exists(source_path):
    print("Source folder not found. Mount Drive (Section 1), set SOURCE_FOLDER_ID (or SOURCE_FOLDER path), and run Section 2.")
else:
    # List last 10 folders (subfolders) in the source folder, by modification time (most recent first)
    subdirs = [f for f in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, f))]
    subdirs_with_mtime = [(d, os.path.getmtime(os.path.join(source_path, d))) for d in subdirs]
    subdirs_with_mtime.sort(key=lambda x: x[1], reverse=True)
    last_10 = subdirs_with_mtime[:10]
    print("Last 10 folders in source folder (most recent first):")
    for name, mtime in last_10:
        dt = datetime.fromtimestamp(mtime).strftime("%Y-%m-%d %H:%M")
        print(f"  {name}  ({dt})")
    if not last_10:
        print("  (no subfolders)")
    print()
    SOURCE_FOLDER = source_path  # for Section 4 fallback

    files = []
    for f in os.listdir(source_path):
        path = os.path.join(source_path, f)
        if os.path.isfile(path) and os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS:
            files.append(path)
    files.sort(key=lambda p: os.path.basename(p))

    if len(files) < IMAGES_PER_GRID:
        print(f"Need at least {IMAGES_PER_GRID} images. Found {len(files)} in {SOURCE_FOLDER}")
    else:
        if SHUFFLE_IMAGES:
            random.shuffle(files)
        max_pages = len(files) // IMAGES_PER_GRID
        num_pages = NUM_GRID_PAGES if NUM_GRID_PAGES and NUM_GRID_PAGES > 0 else max_pages
        num_pages = min(num_pages, max_pages)

        cell_w = (GRID_SIZE[0] - PADDING * (COLS + 1)) // COLS
        cell_h = (GRID_SIZE[1] - PADDING * (ROWS + 1)) // ROWS
        grid_images = []

        for page in range(num_pages):
            base = Image.new("RGB", GRID_SIZE, (255, 255, 255))
            for row in range(ROWS):
                for col in range(COLS):
                    idx = page * IMAGES_PER_GRID + row * COLS + col
                    if idx >= len(files):
                        break
                    try:
                        img = Image.open(files[idx]).convert("RGB")
                        img = ImageOps.fit(img, (cell_w, cell_h), Image.Resampling.LANCZOS)
                        x = PADDING + col * (cell_w + PADDING)
                        y = PADDING + row * (cell_h + PADDING)
                        base.paste(img, (x, y))
                    except Exception as e:
                        print(f"Skip {files[idx]}: {e}")
            grid_images.append(base)

        from IPython.display import display, HTML
        for i, g in enumerate(grid_images):
            display(HTML(f"<h4>Grid {i+1}</h4>"))
            display(g)

        print(f"Built {len(grid_images)} grid(s) from {len(files)} image(s). Run Section 4 to save to Drive.")

## 4. Save grids to Google Drive

Writes the grid images to a new folder on your mounted Drive. You will be **asked for the new folder name** where grids will be saved. The folder is created inside **GOOGLE_DRIVE_PARENT_FOLDER_ID** (or next to the source folder if parent ID is empty).

In [ ]:
try:
    _ = grid_images
except NameError:
    grid_images = []

if not grid_images:
    print("Run Section 3 first to build grids.")
else:
    output_name = input("Name for the new folder where grids will be saved: ").strip()
    output_name = output_name.replace("/", "_").replace("\\", "_") if output_name else "Grids"
    if not output_name:
        output_name = "Grids"
    parent_id = (GOOGLE_DRIVE_PARENT_FOLDER_ID or "").strip()
    if parent_id and os.path.exists("/content/drive/MyDrive"):
        token = _drive_get_access_token()
        parent_path = _drive_folder_id_to_mount_path(token, parent_id)
        if parent_path is not None:
            base = "/content/drive/MyDrive"
            out_path = os.path.join(base, parent_path, output_name)
        else:
            parent = os.path.dirname(SOURCE_FOLDER) if (SOURCE_FOLDER or "").strip() else "/content/drive/MyDrive"
            out_path = os.path.join(parent, output_name)
    else:
        parent = os.path.dirname(SOURCE_FOLDER) if (SOURCE_FOLDER or "").strip() else "/content/drive/MyDrive"
        out_path = os.path.join(parent, output_name)
    os.makedirs(out_path, exist_ok=True)
    n = len(grid_images)
    for i, g in enumerate(grid_images):
        fpath = os.path.join(out_path, f"grid_{i+1:03d}.png")
        g.save(fpath, format="PNG")
    print(f"Saved {n} grid(s) to: {out_path}")
    print(f"Open in Drive: {out_path}")